# YOLOv8 Ball Detection Fine-Tuning - Google Colab

This notebook fine-tunes YOLOv8 for basketball ball/player detection on DeepSport dataset.

## Setup


In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")


In [ ]:
# Install dependencies
%pip install ultralytics -q


## Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Load Dataset


In [ ]:
# Extract dataset from Drive (if uploaded as zip)
import zipfile
import os

dataset_zip = '/content/drive/MyDrive/SHOOTRZ_Datasets/deepsport_yolo.zip'
dataset_path = '/content/datasets/deepsport_yolo'

if os.path.exists(dataset_zip):
    print("Extracting dataset...")
    with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall('/content/datasets')
    print(f"Dataset extracted to {dataset_path}")
else:
    print(f"Dataset zip not found at {dataset_zip}")
    print("Please upload deepsport_yolo.zip to Google Drive: SHOOTRZ_Datasets/")
    
# Verify dataset structure
if os.path.exists(dataset_path):
    print("\nDataset structure:")
    !ls -la {dataset_path}
    print("\nTrain images:")
    !ls {dataset_path}/train/images | head -5


In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Training parameters
config = {
    'model': 'yolov8n.pt',  # Base model
    'data': f'{dataset_path}/data.yaml',
    'epochs': 100,
    'imgsz': 640,
    'batch': 32,  # GPU can handle larger batch
    'device': 0,  # GPU
    'name': 'yolov8n_basketball_deepsport',
    'project': '/content/runs/detect',
    'patience': 20,  # Early stopping
    'save': True,
    'save_period': 10,  # Save checkpoint every 10 epochs
    'lr0': 0.01,  # Initial learning rate
    'lrf': 0.1,   # Final learning rate (lr0 * lrf)
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'box': 7.5,   # Box loss gain
    'cls': 0.5,   # Class loss gain
    'dfl': 1.5,   # DFL loss gain
    'hsv_h': 0.015,  # Image HSV-Hue augmentation
    'hsv_s': 0.7,    # Image HSV-Saturation augmentation
    'hsv_v': 0.4,    # Image HSV-Value augmentation
    'degrees': 0.0,  # Image rotation (+/- deg)
    'translate': 0.1,  # Image translation (+/- fraction)
    'scale': 0.5,     # Image scale (+/- gain)
    'shear': 0.0,     # Image shear (+/- deg)
    'perspective': 0.0,  # Image perspective (+/- fraction)
    'flipud': 0.0,    # Image flip up-down (probability)
    'fliplr': 0.5,    # Image flip left-right (probability)
    'mosaic': 1.0,    # Image mosaic (probability)
    'mixup': 0.0,     # Image mixup (probability)
    'copy_paste': 0.0,  # Segment copy-paste (probability)
}

print("Training configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")


In [ ]:
# Load model
model = YOLO(config['model'])

# Train
print("Starting training...")
results = model.train(**config)

print("\nTraining complete!")


## Validation


In [ ]:
# Validate on test set
metrics = model.val()

print(f"\n=== Validation Results ===")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"\nPer-class mAP@0.5:")
for i, class_name in enumerate(['ball', 'player']):
    if i < len(metrics.box.maps50):
        print(f"  {class_name}: {metrics.box.maps50[i]:.4f}")


## Save to Google Drive


In [ ]:
import shutil

# Copy best model to Drive
drive_models_dir = '/content/drive/MyDrive/SHOOTRZ_Models'
os.makedirs(drive_models_dir, exist_ok=True)

best_model = f"/content/runs/detect/{config['name']}/weights/best.pt"
drive_model_path = f"{drive_models_dir}/yolov8n_basketball_deepsport.pt"

if os.path.exists(best_model):
    shutil.copy(best_model, drive_model_path)
    print(f"Model saved to Drive: {drive_model_path}")
    
    # Also save ONNX if exported
    onnx_path = best_model.replace('.pt', '.onnx')
    if os.path.exists(onnx_path):
        drive_onnx_path = f"{drive_models_dir}/yolov8n_basketball_deepsport.onnx"
        shutil.copy(onnx_path, drive_onnx_path)
        print(f"ONNX model saved to Drive: {drive_onnx_path}")
else:
    print(f"Best model not found: {best_model}")


## Download Model


In [ ]:
# Download model to local machine
from google.colab import files

best_model = f"/content/runs/detect/{config['name']}/weights/best.pt"

if os.path.exists(best_model):
    files.download(best_model)
    print("Model downloaded!")
    
    # Also download ONNX if available
    onnx_path = best_model.replace('.pt', '.onnx')
    if os.path.exists(onnx_path):
        files.download(onnx_path)
        print("ONNX model downloaded!")
else:
    print(f"Best model not found: {best_model}")
